In [1]:
import pandas as pd
import numpy as np

# Load all 4 core datasets
ratings = pd.read_csv('../data/raw/rating.csv')
movies = pd.read_csv('../data/raw/movie.csv')
tags = pd.read_csv('../data/raw/tag.csv')
links = pd.read_csv('../data/raw/link.csv')

print("All datasets loaded.")

All datasets loaded.


In [2]:
def missing_value_report(df, name):
    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100
    report = pd.DataFrame({
        'missing_count': missing_count,
        'missing_percent': missing_percent.round(3)
    })
    report = report[report['missing_count'] > 0]  # only show columns WITH missing values
    print(f"\n--- {name} ---")
    print(f"Total rows: {len(df):,}")
    if report.empty:
        print("No missing values found.")
    else:
        print(report)

missing_value_report(ratings, "ratings")
missing_value_report(movies, "movies")
missing_value_report(tags, "tags")
missing_value_report(links, "links")


--- ratings ---
Total rows: 20,000,263
No missing values found.

--- movies ---
Total rows: 27,278
No missing values found.

--- tags ---
Total rows: 465,564
     missing_count  missing_percent
tag             16            0.003

--- links ---
Total rows: 27,278
        missing_count  missing_percent
tmdbId            252            0.924


In [3]:
# 1. Check for exact duplicate rows (entire row identical)
exact_duplicates = ratings.duplicated().sum()
print(f"Exact duplicate rows in ratings: {exact_duplicates:,}")

# 2. Check for duplicate user-movie pairs (same user rated same movie more than once)
user_movie_duplicates = ratings.duplicated(subset=['userId', 'movieId']).sum()
print(f"Duplicate user-movie rating pairs: {user_movie_duplicates:,}")

# 3. Same checks for tags
exact_tag_duplicates = tags.duplicated().sum()
print(f"\nExact duplicate rows in tags: {exact_tag_duplicates:,}")

user_movie_tag_duplicates = tags.duplicated(subset=['userId', 'movieId', 'tag']).sum()
print(f"Duplicate user-movie-tag entries: {user_movie_tag_duplicates:,}")

Exact duplicate rows in ratings: 0
Duplicate user-movie rating pairs: 0

Exact duplicate rows in tags: 0
Duplicate user-movie-tag entries: 0


In [4]:
# --- RATINGS ---
print("--- ratings ---")
print(f"Exact duplicate rows: {ratings.duplicated().sum():,}")
print(f"Duplicate user-movie pairs: {ratings.duplicated(subset=['userId', 'movieId']).sum():,}")

# --- MOVIES ---
print("\n--- movies ---")
print(f"Exact duplicate rows: {movies.duplicated().sum():,}")
print(f"Duplicate movieId values: {movies.duplicated(subset=['movieId']).sum():,}")
print(f"Duplicate titles (same movie listed twice?): {movies.duplicated(subset=['title']).sum():,}")

# --- TAGS ---
print("\n--- tags ---")
print(f"Exact duplicate rows: {tags.duplicated().sum():,}")
print(f"Duplicate user-movie-tag entries: {tags.duplicated(subset=['userId', 'movieId', 'tag']).sum():,}")

# --- LINKS ---
print("\n--- links ---")
print(f"Exact duplicate rows: {links.duplicated().sum():,}")
print(f"Duplicate movieId values: {links.duplicated(subset=['movieId']).sum():,}")

--- ratings ---
Exact duplicate rows: 0
Duplicate user-movie pairs: 0

--- movies ---
Exact duplicate rows: 0
Duplicate movieId values: 0
Duplicate titles (same movie listed twice?): 16

--- tags ---
Exact duplicate rows: 0
Duplicate user-movie-tag entries: 0

--- links ---
Exact duplicate rows: 0
Duplicate movieId values: 0


In [6]:
# Find the actual duplicate titles and show all their rows together
duplicate_titles = movies[movies.duplicated(subset=['title'], keep=False)].sort_values('title')
print(duplicate_titles)

       movieId                                title  \
20923   102190  20,000 Leagues Under the Sea (1997)   
24064   114130  20,000 Leagues Under the Sea (1997)   
582        588                       Aladdin (1992)   
24092   114240                       Aladdin (1992)   
21429   104035                       Beneath (2013)   
24437   115777                       Beneath (2013)   
13417    66140                      Blackout (2007)   
16827    85070                      Blackout (2007)   
10694    42015                      Casanova (2005)   
26808   128862                      Casanova (2005)   
11205    47254                         Chaos (2005)   
13572    67459                         Chaos (2005)   
21465   104155                 Clear History (2013)   
25886   122940                 Clear History (2013)   
18731    93279                       Darling (2007)   
27064   130062                       Darling (2007)   
823        838                          Emma (1996)   
9135     2

In [7]:
print("--- ratings dtypes ---")
print(ratings.dtypes)

print("\n--- movies dtypes ---")
print(movies.dtypes)

print("\n--- tags dtypes ---")
print(tags.dtypes)

print("\n--- links dtypes ---")
print(links.dtypes)

--- ratings dtypes ---
userId         int64
movieId        int64
rating       float64
timestamp        str
dtype: object

--- movies dtypes ---
movieId    int64
title        str
genres       str
dtype: object

--- tags dtypes ---
userId       int64
movieId      int64
tag            str
timestamp      str
dtype: object

--- links dtypes ---
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object


In [8]:
print("--- Rating Validation ---")
print(f"Minimum rating: {ratings['rating'].min()}")
print(f"Maximum rating: {ratings['rating'].max()}")
print(f"\nUnique rating values (sorted):")
print(sorted(ratings['rating'].unique()))

# Check for any ratings outside the expected 0.5 - 5.0 range
invalid_ratings = ratings[(ratings['rating'] < 0.5) | (ratings['rating'] > 5.0)]
print(f"\nNumber of invalid ratings (outside 0.5-5.0): {len(invalid_ratings):,}")

--- Rating Validation ---
Minimum rating: 0.5
Maximum rating: 5.0

Unique rating values (sorted):
[np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]

Number of invalid ratings (outside 0.5-5.0): 0


In [9]:
# Step 1: Convert timestamp from text to real datetime
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'])
tags['timestamp'] = pd.to_datetime(tags['timestamp'])

# Step 2: Validate the converted date range
print("--- Ratings Timestamp Range ---")
print(f"Earliest rating: {ratings['timestamp'].min()}")
print(f"Latest rating: {ratings['timestamp'].max()}")

print("\n--- Tags Timestamp Range ---")
print(f"Earliest tag: {tags['timestamp'].min()}")
print(f"Latest tag: {tags['timestamp'].max()}")

# Step 3: Check for any suspicious future-dated entries
import datetime
today = pd.Timestamp.now()
future_ratings = ratings[ratings['timestamp'] > today]
print(f"\nRatings with future timestamps: {len(future_ratings):,}")

--- Ratings Timestamp Range ---
Earliest rating: 1995-01-09 11:46:44
Latest rating: 2015-03-31 06:40:02

--- Tags Timestamp Range ---
Earliest tag: 2005-12-24 13:00:10
Latest tag: 2015-03-31 03:09:12

Ratings with future timestamps: 0


In [10]:
# 1. How many movies have "(no genres listed)"?
no_genre_count = (movies['genres'] == '(no genres listed)').sum()
print(f"Movies with '(no genres listed)': {no_genre_count:,}")
print(f"Percentage of total: {(no_genre_count / len(movies)) * 100:.2f}%")

# 2. Extract every individual genre that appears anywhere in the dataset
all_genres = movies['genres'].str.split('|').explode()
unique_genres = sorted(all_genres.unique())

print(f"\nTotal unique genre labels found: {len(unique_genres)}")
print("All genre labels:")
print(unique_genres)

Movies with '(no genres listed)': 246
Percentage of total: 0.90%

Total unique genre labels found: 20
All genre labels:
['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [11]:
# Drop the 16 rows with missing tag text
tags_clean = tags.dropna(subset=['tag']).copy()

print(f"Original tags row count: {len(tags):,}")
print(f"Cleaned tags row count: {len(tags_clean):,}")
print(f"Rows removed: {len(tags) - len(tags_clean):,}")

Original tags row count: 465,564
Cleaned tags row count: 465,548
Rows removed: 16


In [12]:
# Apply the one cleaning change we decided on (tags), 
# and prepare the rest with their datetime conversions already applied

ratings_clean = ratings.copy()      # already has timestamp converted from Task 5
movies_clean = movies.copy()        # no row-level changes needed (Task 7 decision)
tags_clean = tags_clean             # already created in Task 7 (16 rows dropped)
links_clean = links.copy()          # no row-level changes needed (Task 7 decision)

# Save all 4 cleaned datasets to data/processed/
ratings_clean.to_csv('../data/processed/ratings_clean.csv', index=False)
movies_clean.to_csv('../data/processed/movies_clean.csv', index=False)
tags_clean.to_csv('../data/processed/tags_clean.csv', index=False)
links_clean.to_csv('../data/processed/links_clean.csv', index=False)

print("All cleaned datasets saved successfully to data/processed/")

All cleaned datasets saved successfully to data/processed/
